## LangChain Layer 1 Excercise: The Fictional Weather Report
Here we will show fictional weather reports for 3 cities
                               

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

In [ ]:
llm = ChatOpenAI(model ="gpt-5.4-mini")
message = "Define langchain kill switch in 2 lines"

response = llm.invoke(message)
print(response.content)

In [ ]:
@tool
def get_city_temperature(symbol: str) -> int:
    """This will provide the temperature of a city"""
    fake_temperatures = {"Delhi":25, "New York":-5, "Dubai":45}
    return fake_temperatures.get(symbol)
    
print("name:", get_city_temperature.name)
print("description:", get_city_temperature.description)
print("args:", get_city_temperature.args)
print("City Temperature:", get_city_temperature.invoke({"symbol": "Dubai"}))

In [ ]:
llm_with_tool = llm.bind_tools([get_city_temperature])

response = llm_with_tool.invoke("what is the temperature of Dubai?")
print("content:", (response.content))
print("tool_calls:", response.tool_calls)

Manual tool Run

In [ ]:
conversation = [HumanMessage("what is the temperature of New York?")]
ai_call = llm_with_tool.invoke(conversation)
conversation.append(ai_call)

for call in ai_call.tool_calls:
    if call["name"] == "get_city_temperature":
        result = get_city_temperature.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

final = llm_with_tool.invoke(conversation)
print(final.content)

In [ ]:
class City(BaseModel):
    name: str = Field(description="The City name")
    temperature: int = Field(description="The current temperature of the City")
    current_weather: str = Field(description="The running weather condition of the city")

structured_llm = llm.with_structured_output(City)
city = structured_llm.invoke("Tell me about the Dubai")
print(city)
print("weather condition: ", city.current_weather)